# Block 1 — Mouth Aspect Ratio (MAR) for yawning

In [1]:
import numpy as np

# Mouth landmark points
MOUTH = [61, 291, 39, 181, 0, 17]

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def calculate_mar(landmarks, mouth_points, w, h):
    pts = [(int(landmarks[i].x * w),
            int(landmarks[i].y * h)) for i in mouth_points]
    A = distance(pts[1], pts[5])
    B = distance(pts[2], pts[4])
    C = distance(pts[0], pts[3])
    return round((A + B) / (2.0 * C), 3)

print("MAR function ready!")

#Exactly same formula as EAR. When mouth opens wide — MAR goes above 0.60 — yawning detected.

MAR function ready!


# Block 2 — Head angle for distraction

In [2]:
import numpy as np

# Nose tip, chin, left ear, right ear landmark indices
NOSE_TIP  = 1
CHIN      = 152
LEFT_EAR  = 234
RIGHT_EAR = 454


def calculate_head_angle(landmarks,w,h):
    nose=np.array([landmarks[NOSE_TIP].x * w , landmarks[NOSE_TIP].y * h])
    l_ear = np.array([landmarks[LEFT_EAR].x  * w, landmarks[LEFT_EAR].y  * h])
    r_ear = np.array([landmarks[RIGHT_EAR].x * w, landmarks[RIGHT_EAR].y * h])

    # Middle point between ears
    mid   = (l_ear + r_ear) / 2

    # Vector from mid to nose
    vec   = nose - mid

    angle=np.degrees(np.arctan2(vec[0],vec[1]))
    return abs(round(angle,1))

print("Head angle function ready!")



Head angle function ready!


# Block 3 — Safety score

In [3]:
# Starts at 100, goes down with each event
class SafetyScore:
    def __init__(self):
        self.score=100
        self.events=[]

    def deduct(self,event_type):
        deductions={
            "DROWSY":     10,
            "YAWNING":     5,
            "DISTRACTED":  7,
            "MICROSLEEP": 15 
        }

        points=deductions.get(event_type,5)
        self.score=max(0,self.score-points)
        self.events.append(event_type)
        print(f"Event: {event_type} → Score: {self.score}")

    def get_risk(self):
        if self.score>=70:
            return "Safe", (0,255,0)
        elif self.score>=40:
            return "Warning",(0,165,255)
        else:
            return "Danger", (0,0,255)

# Test it
safety = SafetyScore()
safety.deduct("DROWSY")
safety.deduct("YAWNING")
print(safety.get_risk())


Event: DROWSY → Score: 90
Event: YAWNING → Score: 85
('Safe', (0, 255, 0))


# Block 4 — Complete detection system

In [7]:
import cv2
import mediapipe as mp
import numpy as np

mp_face_mesh = mp.solutions.face_mesh
face_mesh    = mp_face_mesh.FaceMesh(
    max_num_faces            = 1,
    min_detection_confidence = 0.5,
    min_tracking_confidence  = 0.5
)

# Landmark indices
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33,  160, 158, 133, 153, 144]

# ✅ FIX: Simpler and correct mouth points for MAR
MOUTH_TOP    = 13   # top of inner lip
MOUTH_BOTTOM = 14   # bottom of inner lip
MOUTH_LEFT   = 78   # left corner
MOUTH_RIGHT  = 308  # right corner

NOSE_TIP  = 1
LEFT_EAR  = 234
RIGHT_EAR = 454

# Thresholds
EAR_THRESHOLD  = 0.28
MAR_THRESHOLD  = 0.4   # ✅ FIX: lower threshold for simpler MAR formula
HEAD_THRESHOLD = 30

# Frame limits
DROWSY_LIMIT   = 30
YAWN_LIMIT     = 20
DISTRACT_LIMIT = 90

# Counters
drowsy_counter   = 0
yawn_counter     = 0
distract_counter = 0
safety_score     = 100

# ✅ FIX: Cooldown flags — prevent repeated deductions per episode
drowsy_cooldown   = False
yawn_cooldown     = False
distract_cooldown = False


def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))


def calculate_ear(landmarks, eye_pts, w, h):
    pts = [(int(landmarks[i].x * w),
            int(landmarks[i].y * h)) for i in eye_pts]
    A = distance(pts[1], pts[5])
    B = distance(pts[2], pts[4])
    C = distance(pts[0], pts[3])
    return round((A + B) / (2.0 * C), 3)


# ✅ FIX: Simpler MAR — vertical / horizontal distance
# When mouth closed → vertical small → MAR low (~0.1-0.2)
# When mouth open   → vertical large → MAR high (~0.5-0.7)
def calculate_mar(landmarks, w, h):
    top    = (int(landmarks[MOUTH_TOP].x    * w), int(landmarks[MOUTH_TOP].y    * h))
    bottom = (int(landmarks[MOUTH_BOTTOM].x * w), int(landmarks[MOUTH_BOTTOM].y * h))
    left   = (int(landmarks[MOUTH_LEFT].x   * w), int(landmarks[MOUTH_LEFT].y   * h))
    right  = (int(landmarks[MOUTH_RIGHT].x  * w), int(landmarks[MOUTH_RIGHT].y  * h))

    vertical   = distance(top, bottom)
    horizontal = distance(left, right)

    return round(vertical / horizontal, 3)


def calculate_head_angle(landmarks, w, h):
    nose  = np.array([landmarks[NOSE_TIP].x * w,
                      landmarks[NOSE_TIP].y * h])
    l_ear = np.array([landmarks[LEFT_EAR].x * w,
                      landmarks[LEFT_EAR].y * h])
    r_ear = np.array([landmarks[RIGHT_EAR].x * w,
                      landmarks[RIGHT_EAR].y * h])
    mid   = (l_ear + r_ear) / 2
    vec   = nose - mid
    return abs(round(np.degrees(np.arctan2(vec[0], vec[1])), 1))


def get_risk(score):
    if score >= 70:   return "Safe",    (0, 255, 0)
    elif score >= 40: return "Warning", (0, 165, 255)
    else:             return "Danger",  (0, 0, 255)


cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break

    h, w    = frame.shape[:2]
    rgb     = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    # Reset alerts each frame
    drowsy_alert   = False
    yawn_alert     = False
    distract_alert = False

    if results.multi_face_landmarks:
        landmarks = results.multi_face_landmarks[0].landmark

        # Calculate metrics
        left_ear  = calculate_ear(landmarks, LEFT_EYE,  w, h)
        right_ear = calculate_ear(landmarks, RIGHT_EYE, w, h)
        avg_ear   = round((left_ear + right_ear) / 2, 3)
        mar       = calculate_mar(landmarks, w, h)   # ✅ FIX: no mouth_pts arg needed
        angle     = calculate_head_angle(landmarks, w, h)

        # ── Drowsiness check ──────────────────────────
        if avg_ear < EAR_THRESHOLD:
            drowsy_counter += 1
            # ✅ FIX: cooldown prevents repeated deduction in same episode
            if drowsy_counter >= DROWSY_LIMIT and not drowsy_cooldown:
                drowsy_alert    = True
                safety_score    = max(0, safety_score - 10)
                drowsy_cooldown = True   # block further deductions
                drowsy_counter  = 0
                print("DROWSY event logged")
        else:
            drowsy_counter  = 0
            drowsy_alert    = False
            drowsy_cooldown = False  # ✅ FIX: reset cooldown when eyes open

        # ── Yawning check ─────────────────────────────
        if mar > MAR_THRESHOLD:
            yawn_counter += 1
            # ✅ FIX: cooldown prevents repeated deduction in same yawn
            if yawn_counter >= YAWN_LIMIT and not yawn_cooldown:
                yawn_alert    = True
                safety_score  = max(0, safety_score - 5)
                yawn_cooldown = True
                yawn_counter  = 0
                print("YAWN event logged")
        else:
            yawn_counter  = 0
            yawn_alert    = False
            yawn_cooldown = False  # ✅ FIX: reset cooldown when mouth closes

        # ── Distraction check ─────────────────────────
        if angle > HEAD_THRESHOLD:
            distract_counter += 1
            # ✅ FIX: cooldown prevents repeated deduction for same distraction
            if distract_counter >= DISTRACT_LIMIT and not distract_cooldown:
                distract_alert    = True
                safety_score      = max(0, safety_score - 7)
                distract_cooldown = True
                distract_counter  = 0
                print("DISTRACTED event logged")
        else:
            distract_counter  = 0
            distract_alert    = False
            distract_cooldown = False  # ✅ FIX: reset cooldown when head straight

        # ── Risk level ────────────────────────────────
        risk_label, risk_color = get_risk(safety_score)

        # ── Display metrics ───────────────────────────
        cv2.putText(frame, f"EAR: {avg_ear}",          (20, 35),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"MAR: {mar}",               (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Angle: {angle}",           (20, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Score: {safety_score}/100",(20, 125),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, f"Risk: {risk_label}",       (20, 155),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, risk_color, 2)

        # ── Alerts ────────────────────────────────────
        if drowsy_alert:
            cv2.putText(frame, "DROWSY!", (w//2 - 80, 60),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 0, 255), 3)
            cv2.rectangle(frame, (0, 0), (w, h), (0, 0, 255), 5)

        if yawn_alert:
            cv2.putText(frame, "YAWNING!", (w//2 - 90, 110),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 165, 255), 3)

        if distract_alert:
            cv2.putText(frame, "DISTRACTED!", (w//2 - 110, 160),
                        cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 0, 0), 3)

    cv2.imshow("Driver Safety Monitor", frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

DROWSY event logged
DROWSY event logged
DROWSY event logged
DROWSY event logged
DROWSY event logged
DROWSY event logged
